In [221]:
import pandas as pd
from scipy.spatial import KDTree
from tqdm import tqdm

## Do it via the known output:

In [222]:
cf = pd.read_pickle("../2_clear_cases/clear-fusion_tama_augustus.pkl")
cf["gene_id"]=cf["accession"]+"_"+cf["gene"]
cf["isoform_id"]=cf["accession"]+"_"+cf["isoform"]

In [223]:
lengths=pd.DataFrame()
for acc in tqdm(cf["accession"].unique().tolist()):
    # -------------1-------------
    # prepare sqanti output
    sqanti = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{}_augustus_sqanti/{}-tama-vs-augustus_classification.txt".format(acc,acc), sep="\t")
    sqanti["gene"] = sqanti["isoform"].str.split(".").str[0]
    sqanti["isoform_id"] = acc+"_"+sqanti["isoform"]
    sub = sqanti[["ORF_length","CDS_length","isoform_id"]]
    lengths=pd.concat([lengths,sub])

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:38<00:00,  2.12s/it]


In [224]:
cf=pd.merge(cf, lengths, on=["isoform_id"], how="left")

In [225]:
cf

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,...,cat_read_support,gene_association_read_support,type,isoforms_per_gene,ORFs_per_gene,isoforms_per_ORFs,gene_id,isoform_id,ORF_length,CDS_length
0,G1128,G1128.10,at6923,g1555,novel,1,3275,1,59,genic,...,59,12,nlr,11,2,8.0,at6923_G1128,at6923_G1128.10,393.0,1182.0
1,G1128,G1128.11,at6923,g1555,novel,1,2744,1,59,genic,...,59,12,nlr,11,2,3.0,at6923_G1128,at6923_G1128.11,395.0,1188.0
2,G1128,G1128.12,at6923,g1555,novel,1,2795,1,59,genic,...,59,12,nlr,11,2,3.0,at6923_G1128,at6923_G1128.12,395.0,1188.0
3,G1128,G1128.13,at6923,g1555,novel,1,1821,1,59,genic,...,59,12,nlr,11,2,3.0,at6923_G1128,at6923_G1128.13,395.0,1188.0
4,G1128,G1128.2,at6923,g1555,novel,2,3331,1,59,novel_not_in_catalog,...,59,12,nlr,11,2,8.0,at6923_G1128,at6923_G1128.2,393.0,1182.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3408192,G16302,G16302.24,col0,ATCG01310_ATCG01300,novel,2,818,1,136,fusion,...,15,15,other,49,8,2.0,col0_G16302,col0_G16302.24,253.0,759.0
3408193,G16302,G16302.26,col0,ATCG01310_ATCG01300,novel,2,831,2,136,fusion,...,15,15,other,49,8,4.0,col0_G16302,col0_G16302.26,257.0,771.0
3408194,G16302,G16302.5,col0,ATCG01310_ATCG01300,novel,2,1204,1,136,fusion,...,15,15,other,49,8,4.0,col0_G16302,col0_G16302.5,257.0,771.0
3408195,G16302,G16302.7,col0,ATCG01310_ATCG01300,novel,2,1141,6,136,fusion,...,15,15,other,49,8,4.0,col0_G16302,col0_G16302.7,257.0,771.0


## Fast KDtree solution

In [232]:
PCA_genes = pd.read_pickle("../7_PCA/PCA_genes.pkl")
PCA_genes["accession"]=PCA_genes["gene"].str.split("_").str[0]
PCA_genes.fillna(0, inplace=True)

In [250]:
PCA_genes["accession"].unique()

array(['at6923', 'at6929', 'at7143', 'at8285', 'at9104', 'at9336',
       'at9503', 'at9578', 'at9744', 'at9762', 'at9806', 'at9830',
       'at9847', 'at9852', 'at9879', 'at9883', 'at9900', 'col0'],
      dtype=object)

In [234]:
PCA_genes.columns

Index(['exons', 'length', 'genic', 'novel_not_in_catalog', 'full-splice_match',
       'incomplete-splice_match', 'novel_in_catalog', 'fusion', 'mono-exon',
       'intron_retention', 'alternative_3end5end', '3prime_fragment',
       'at_least_one_novel_splicesite', 'mono-exon_by_intron_retention',
       '5prime_fragment', 'alternative_3end', 'internal_fragment',
       'alternative_5end', 'multi-exon', 'reference_match',
       'combination_of_known_splicesites', 'combination_of_known_junctions',
       'n_ORFs', 'n_isoforms', 'n_noncoding', 'ORF_length', 'CDS_length',
       'gene', 'type', 'accession'],
      dtype='object')

In [235]:
## create a second run with only sqanti categories
#PCA_genes = PCA_genes[['genic', 'novel_not_in_catalog', 'full-splice_match',
#       'incomplete-splice_match', 'novel_in_catalog', 'fusion', 'mono-exon',
#       'intron_retention', 'alternative_3end5end', '3prime_fragment',
#       'at_least_one_novel_splicesite', 'mono-exon_by_intron_retention',
#       '5prime_fragment', 'alternative_3end', 'internal_fragment',
#       'alternative_5end', 'multi-exon', 'reference_match',
#       'combination_of_known_splicesites', 'combination_of_known_junctions',
#       'n_ORFs', 'n_isoforms', 'n_noncoding',
#       'gene', 'type', 'accession']]

In [236]:
## create a second run withput CDS and ORF
PCA_genes = PCA_genes[['exons', 'length', 'genic', 'novel_not_in_catalog', 'full-splice_match',
       'incomplete-splice_match', 'novel_in_catalog', 'fusion', 'mono-exon',
       'intron_retention', 'alternative_3end5end', '3prime_fragment',
       'at_least_one_novel_splicesite', 'mono-exon_by_intron_retention',
       '5prime_fragment', 'alternative_3end', 'internal_fragment',
       'alternative_5end', 'multi-exon', 'reference_match',
       'combination_of_known_splicesites', 'combination_of_known_junctions',
       'n_ORFs', 'n_isoforms', 'n_noncoding',
       'gene', 'type', 'accession']]

In [273]:
nn_all=pd.DataFrame()
gene_dict={}
for acc in PCA_genes["accession"].unique().tolist():
    # get acc subsets by type
    acc_df=PCA_genes.loc[PCA_genes["accession"]==acc]
    # hande NLR
    nlr_values=acc_df.loc[acc_df["type"]=="nlr"].drop(["accession","gene","type"], axis=1)
    nlr_values.reset_index(drop=True, inplace=True)
    nlr_labels=acc_df.loc[acc_df["type"]=="nlr"][["accession","gene","type"]]
    nlr_labels.reset_index(drop=True, inplace=True)
    
    other_values=acc_df.loc[~(acc_df["type"]=="nlr")].drop(["accession","gene","type"], axis=1)
    other_values.reset_index(drop=True, inplace=True)
    other_labels=acc_df.loc[~(acc_df["type"]=="nlr")][["accession","gene","type"]]
    other_labels.reset_index(drop=True, inplace=True)
    
    # build tree
    kdB = KDTree(other_values)
    nn= kdB.query(nlr_values, k=5)[-1]
    
    # get corresponding genes from other_labels
    ids=[]
    for i in nn:
        genes=[]
        for g in i:
            genes.append(other_labels.iloc[g]["gene"].split("_")[1])

        ids.append(genes)
    
    # create full df per acc
    genes_df=pd.DataFrame(ids)
    genes_df=pd.concat([nlr_labels, genes_df], axis=1)
    nn_all=pd.concat([nn_all, genes_df])
    
    # create dict entry per acc
    gene_dict[acc]= set([g for sublist in ids for g in sublist])

126
630
565
117
585
534
124
620
566
118
590
520
124
620
579
124
620
558
128
640
581
129
645
569
138
690
584
130
650
589
130
650
560
116
580
526
132
660
588
125
625
535
118
590
513
125
625
571
127
635
579
123
615
568


In [238]:
len(gene_dict["at6923"])

565

In [239]:
nn_all.reset_index(inplace=True, drop=True)
nn_all["gene"]=nn_all[["gene"]].apply(lambda x: x.str.split("_").str[1])

In [240]:
## get all isoforms per gene
iso_dict={}
for acc in cf["accession"].unique().tolist():
    isos=[]
    acc_df=cf.loc[cf["accession"]==acc]
    for gene in acc_df.groupby("gene"):
        if gene[0] in gene_dict[acc]:
            isos.extend(gene[1]["isoform"].tolist())
    iso_dict[acc]=isos

In [241]:
def read(file_name):
    try:
        fasta = open(file_name, 'r')
        fasta_pairs = {}
        header = ''
        # check every line for a new header
        # if so, set as key and set non header lines as value
        for line in fasta:
            if line[0] == '>':
                header = line.strip('\n')
                fasta_pairs[header] = ''
            else:
                fasta_pairs[header] += line.strip('\n')
        # return dict of fasta entries
        return fasta_pairs
    except IOError:
        print("File \"{}\" could not be opened.".format(file_name))

In [242]:
for acc in iso_dict.keys():
    fasta = read(f"../../output/v2/collapse_tama/{acc}_transdecoder/{acc}_transcripts.fasta.transdecoder.pep")
    
    f_out=[]
    for g in fasta.keys():
        name = g.replace(">","").split(".")[0] +"."+ g.replace(">","").split(".")[1]
        if name in iso_dict[f"{acc}"]:
            f_out.append(g)
            
    ofile = open(f"input/n5_genes/{acc}_no_co_nlr_like.prot.fasta", "w")

    for g in f_out:
        ofile.write(g + "\n" + fasta[g].replace("*","") + "\n")

    #do not forget to close it
    ofile.close()

## Analyze Pfam based on all parameters

In [276]:
runs = ["","_cat","_no_co"]
for run in runs:
    pfam_nlrs=pd.DataFrame()
    pfam_dom_gene=pd.DataFrame()
    pfam_dom_iso=pd.DataFrame()
    pfam_orf_gene=pd.DataFrame()
    pfam_orf_iso=pd.DataFrame()
    for acc in tqdm(cf["accession"].unique().tolist()):
        # data preparation
        pfam = pd.read_csv(f"output/n5_genes/{acc}{run}/{acc}{run}_nlr_like.prot.fasta.tsv", sep="\t", names=["Query protein","Sequence MD5 digest","Sequence length","Analysis","Signature accession","Signature description","Start location","Stop location","Score (e-value)","Status (T: true)","Date","InterPro annotations - accession","InterPro annotations - description","GOTerms"])
        pfam = pfam.loc[pfam['Analysis']=="Pfam"]
        pfam["gene"]=pfam["Query protein"].str.split(".").str[0]
        pfam["isoform"]=pfam["Query protein"].str.split(".").str[0]+"."+pfam["Query protein"].str.split(".").str[1]

        # n query proteins
        # n query isoforms
        # n query genes

        ## NB-ARC analysis
        # get all transdecoder orfs with NB-ARC relation
        nbarc_trans = pfam.loc[pfam["Signature description"]=="NB-ARC domain"]["Query protein"].tolist()
        # extract full info from Pfam
        nbarc = pfam.loc[pfam["Query protein"].isin(nbarc_trans)]
        # NB-ARC gene:
        nlr_genes_dict={"gene":[],"TIR":[],"LRR":[],"CC":[]}
        #print(nlr_genes_dict)
        for i in nbarc.groupby("gene"):
            # get gene id
            nlr_genes_dict["gene"].append(i[0])

            # get TIR
            if "TIR domain" in i[1]["Signature description"].tolist():
                nlr_genes_dict["TIR"].append(1)
            else:
                nlr_genes_dict["TIR"].append(0)

            # get LRR
            lrr_signature=i[1]["Signature description"]
            lrr=0
            for j in lrr_signature:
                if "Leucine" in j:
                    lrr+=1
                    break
            nlr_genes_dict["LRR"].append(lrr)

            # get CC (RX domain in pfam!)

            cc_signature=i[1]["Signature description"]
            cc=0
            for j in cc_signature:
                if "coil" in j:
                    cc+=1
                    break
            nlr_genes_dict["CC"].append(cc)
        nlr_out=pd.DataFrame.from_dict(nlr_genes_dict, orient="index").transpose()
        nlr_out["accession"]=acc
        pfam_nlrs=pd.concat([pfam_nlrs, nlr_out])


        ## Pfam analysis
        # gene
        dom_gene=pfam[["gene","InterPro annotations - description"]].drop_duplicates()
        dom_gene=dom_gene["InterPro annotations - description"].value_counts()
        dom_gene.rename(acc, inplace=True)
        pfam_dom_gene=pd.concat([pfam_dom_gene, dom_gene.head(n=11)], axis=1, sort=False)

        # isoform
        dom_iso=pfam[["isoform","InterPro annotations - description"]].drop_duplicates()
        dom_iso=dom_iso["InterPro annotations - description"].value_counts()
        dom_iso.rename(acc, inplace=True)
        pfam_dom_iso=pd.concat([pfam_dom_iso, dom_iso.head(n=11)], axis=1, sort=False)

        ## ORF analysis
        # gene
        orf_gene=pfam[["gene","GOTerms"]].drop_duplicates()
        orf_gene["GOTerms_list"]=pfam_gene["GOTerms"].str.split("|")
        orf_gene=orf_gene.explode("GOTerms_list").drop("GOTerms", axis=1).drop_duplicates()["GOTerms_list"].value_counts()
        orf_gene.rename(acc, inplace=True)
        pfam_orf_gene=pd.concat([pfam_orf_gene, orf_gene.head(n=11)], axis=1, sort=False)

        # isoform
        orf_iso=pfam[["isoform","GOTerms"]].drop_duplicates()
        orf_iso["GOTerms_list"]=orf_iso["GOTerms"].str.split("|")
        orf_iso=orf_iso.explode("GOTerms_list").drop("GOTerms", axis=1).drop_duplicates()["GOTerms_list"].value_counts()
        orf_iso.rename(acc, inplace=True)
        pfam_orf_iso=pd.concat([pfam_orf_iso, orf_iso.head(n=11)], axis=1, sort=False)
    pfam_nlrs.to_csv(f"output/n5_genes/pfam{run}_nlrs.tsv", sep="\t")
    pfam_dom_gene.to_csv(f"output/n5_genes/pfam{run}_dom_gene.tsv", sep="\t")
    pfam_dom_iso.to_csv(f"output/n5_genes/pfam{run}_dom_iso.tsv", sep="\t")
    pfam_orf_gene.to_csv(f"output/n5_genes/pfam{run}_orf_gene.tsv", sep="\t")
    pfam_orf_iso.to_csv(f"output/n5_genes/pfam{run}_orf_iso.tsv", sep="\t")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:03<00:00,  5.06it/s]
